# 1. Generate observations from the new probabilistic grammar

This notebook generates **850 random observations** from the updated fictional grammar.

```text
S -> s A | s B | s C
A -> F | r F | r r F | r r e
B -> l A
C -> l l A
F -> f e | p e
```

Because no new production probabilities were specified, the defaults below use a **uniform distribution among alternatives**:

- `S`: 1/3 for each alternative
- `A`: 1/4 for each alternative
- `F`: 1/2 for each alternative
- `B` and `C`: probability 1.0 because each has one alternative

All probabilities are configurable in the next cell.

The output keeps the same format expected by the downstream notebooks: `observations.csv` with columns `id` and `sentence`.


In [1]:
import random
import pandas as pd

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
N_OBSERVATIONS = 850
OUTPUT_FILE = "observations.csv"

# Set to an integer for reproducible demonstrations.
# Set to None for a different dataset on each execution.
SEED = 42

# Production probabilities
PROBABILITIES = {
    "S": [1/3, 1/3, 1/3],
    "A": [0.25, 0.25, 0.25, 0.25],
    "F": [0.50, 0.50],
}

if SEED is not None:
    random.seed(SEED)


In [2]:
# ------------------------------------------------------------
# Validate probabilities
# ------------------------------------------------------------
for nonterminal, probabilities in PROBABILITIES.items():
    total = sum(probabilities)
    if abs(total - 1.0) > 1e-9:
        raise ValueError(
            f"Probabilities for {nonterminal} must sum to 1. "
            f"Current sum: {total}"
        )

print("All configured probability distributions sum to 1.")


All configured probability distributions sum to 1.


In [3]:
# ------------------------------------------------------------
# Probabilistic grammar
# ------------------------------------------------------------
GRAMMAR = {
    "S": [
        (PROBABILITIES["S"][0], ["s", "A"]),
        (PROBABILITIES["S"][1], ["s", "B"]),
        (PROBABILITIES["S"][2], ["s", "C"]),
    ],
    "A": [
        (PROBABILITIES["A"][0], ["F"]),
        (PROBABILITIES["A"][1], ["r", "F"]),
        (PROBABILITIES["A"][2], ["r", "r", "F"]),
        (PROBABILITIES["A"][3], ["r", "r", "e"]),
    ],
    "B": [
        (1.0, ["l", "A"]),
    ],
    "C": [
        (1.0, ["l", "l", "A"]),
    ],
    "F": [
        (PROBABILITIES["F"][0], ["f", "e"]),
        (PROBABILITIES["F"][1], ["p", "e"]),
    ],
}

def choose(options):
    probabilities = [p for p, _ in options]
    productions = [rhs for _, rhs in options]
    return random.choices(productions, weights=probabilities, k=1)[0]

def generate_sentence():
    symbols = ["S"]

    while any(symbol in GRAMMAR for symbol in symbols):
        new_symbols = []
        for symbol in symbols:
            if symbol in GRAMMAR:
                new_symbols.extend(choose(GRAMMAR[symbol]))
            else:
                new_symbols.append(symbol)
        symbols = new_symbols

    return " ".join(symbols)


In [4]:
# ------------------------------------------------------------
# Generate observations
# ------------------------------------------------------------
sentences = [generate_sentence() for _ in range(N_OBSERVATIONS)]

observations = pd.DataFrame({
    "id": range(1, N_OBSERVATIONS + 1),
    "sentence": sentences,
})

observations.to_csv(OUTPUT_FILE, index=False)

print(f"Generated {len(observations)} observations.")
print(f"Saved to: {OUTPUT_FILE}")
observations.head(10)


Generated 850 observations.
Saved to: observations.csv


,id,sentence
0,1,s l r f e
1,2,s l l r r e
2,3,s r f e
3,4,s r r f e
4,5,s r r p e
5,6,s r r p e
6,7,s r r e
7,8,s l l p e
8,9,s l p e
9,10,s l r r p e


## Check the possible sentences

The grammar generates **16 distinct terminal sentences**:

- `S -> s A`: 4 choices from `A`, with 2 choices from `F` except `rr e`, giving 7 sentences
- `S -> s B`: 7 sentences
- `S -> s C`: 7 sentences

Therefore the total is:

\[
7 + 7 + 7 = \boxed{21}
\]

The empirical probabilities should approach the theoretical probabilities as the number of observations increases.


In [5]:
# ------------------------------------------------------------
# Empirical frequencies
# ------------------------------------------------------------
frequency = (
    observations["sentence"]
    .value_counts()
    .rename_axis("sentence")
    .reset_index(name="count")
)

frequency["observed_probability"] = frequency["count"] / N_OBSERVATIONS
frequency


,sentence,count,observed_probability
0,s l l r r e,80,0.094118
1,s r r e,76,0.089412
2,s l r r e,67,0.078824
3,s l r r f e,53,0.062353
4,s l r r p e,49,0.057647
5,s l l f e,43,0.050588
6,s r r p e,43,0.050588
7,s p e,40,0.047059
8,s r r f e,39,0.045882
9,s r f e,37,0.043529


## Theoretical sentence probabilities

For example:

\[
P(srrfe)
=
P(S\to sA)
P(A\to rrF)
P(F\to fe).
\]

With the default probabilities:

\[
P(srrfe)=\frac13\times0.25\times0.50
=0.04167.
\]

The next cell computes the theoretical probability of every sentence generated by the grammar and compares it with the empirical frequency.


In [6]:
# ------------------------------------------------------------
# Compute theoretical probabilities for all terminal sentences
# ------------------------------------------------------------
def add_A_sentences(theoretical, prefix, p_S):
    # A -> F
    theoretical[prefix + ["f", "e"]] = p_S * 0.25 * 0.50
    theoretical[prefix + ["p", "e"]] = p_S * 0.25 * 0.50

    # A -> r F
    theoretical[prefix + ["r", "f", "e"]] = p_S * 0.25 * 0.50
    theoretical[prefix + ["r", "p", "e"]] = p_S * 0.25 * 0.50

    # A -> r r F
    theoretical[prefix + ["r", "r", "f", "e"]] = p_S * 0.25 * 0.50
    theoretical[prefix + ["r", "r", "p", "e"]] = p_S * 0.25 * 0.50

    # A -> r r e
    theoretical[prefix + ["r", "r", "e"]] = p_S * 0.25

theoretical = {}

pS = PROBABILITIES["S"]
add_A_sentences(theoretical, ["s"], pS[0])
add_A_sentences(theoretical, ["s", "l"], pS[1])
add_A_sentences(theoretical, ["s", "l", "l"], pS[2])

theoretical = {
    " ".join(sentence): probability
    for sentence, probability in theoretical.items()
}

comparison = pd.DataFrame({
    "theoretical_probability": pd.Series(theoretical),
    "observed_count": observations["sentence"].value_counts(),
})

comparison["observed_count"] = comparison["observed_count"].fillna(0).astype(int)
comparison["observed_probability"] = (
    comparison["observed_count"] / N_OBSERVATIONS
)

comparison.sort_index()


TypeError: unhashable type: 'list'